In [2]:
import pandas as pd
from pathlib import Path
import sys 

In [ ]:
RUTA_PROYECTO = Path("..")
RUTA_raiz = RUTA_PROYECTO / "data" / "raiz"
RUTA_provisional = RUTA_PROYECTO / "data" / "provisional"

RUTA_provisional.mkdir(parents=True, exist_ok=True)

ruta_resenas = RUTA_raiz / "Video_Games.jsonl.gz"

ruta_metadatos = RUTA_raiz / "meta_Video_Games.jsonl.gz"

print("Archivo de reseñas:", ruta_resenas.exists())
print("Archivo de metadatos:", ruta_metadatos.exists())
print("Carpeta de salida:", RUTA_provisional.resolve())

Archivo de reseñas: True
Archivo de metadatos: True
Carpeta de salida: C:\Users\Paula María\Desktop\IS\5º CURSO\TFG\Trabajo-Fin-de-Grado\data\interim


In [4]:
def es_videojuego(categorias):
    """
    Devuelve True si el valor es una lista
    y contiene el término exacto 'Games'.
    """
    return isinstance(categorias, list) and "Games" in categorias

In [5]:
# Comprobación de la función es_videojuego con ejemplos de prueba

print(es_videojuego(["Video Games", "PC", "Games"]))
print(es_videojuego(["Video Games", "PlayStation 4", "Accessories"]))
print(es_videojuego([]))

True
False
False


In [6]:
columnas_metadatos = [
    "parent_asin",
    "main_category",
    "title",
    "average_rating",
    "rating_number",
    "store",
    "categories"
]

bloques_videojuegos = []

total_productos_revisados = 0
total_productos_seleccionados = 0

In [7]:
#Bucle para leer el archivo de metadatos en bloques y filtrar los videojuegos


for numero_bloque, bloque in enumerate(
    pd.read_json(
        ruta_metadatos,
        lines=True,
        compression="gzip",
        chunksize=1000
    ),
    start=1
):
    total_productos_revisados += len(bloque)

    seleccion = bloque[
        bloque["categories"].apply(es_videojuego)
    ].copy()

    total_productos_seleccionados += len(seleccion)

    # bucle para seleccionar las columnas disponibles en el bloque actual y agregarlas a la lista de bloques de videojuegos

    columnas_disponibles = [
        columna
        for columna in columnas_metadatos
        if columna in seleccion.columns
    ]

    if not seleccion.empty:
        bloques_videojuegos.append(
            seleccion[columnas_disponibles]
        )

    print(
        f"Bloque {numero_bloque}: "
        f"{len(bloque):,} productos revisados, "
        f"{len(seleccion):,} seleccionados"
    )

Bloque 1: 1,000 productos revisados, 334 seleccionados
Bloque 2: 1,000 productos revisados, 346 seleccionados
Bloque 3: 1,000 productos revisados, 317 seleccionados
Bloque 4: 1,000 productos revisados, 354 seleccionados
Bloque 5: 1,000 productos revisados, 349 seleccionados
Bloque 6: 1,000 productos revisados, 333 seleccionados
Bloque 7: 1,000 productos revisados, 318 seleccionados
Bloque 8: 1,000 productos revisados, 317 seleccionados
Bloque 9: 1,000 productos revisados, 348 seleccionados
Bloque 10: 1,000 productos revisados, 356 seleccionados
Bloque 11: 1,000 productos revisados, 319 seleccionados
Bloque 12: 1,000 productos revisados, 304 seleccionados
Bloque 13: 1,000 productos revisados, 349 seleccionados
Bloque 14: 1,000 productos revisados, 360 seleccionados
Bloque 15: 1,000 productos revisados, 359 seleccionados
Bloque 16: 1,000 productos revisados, 347 seleccionados
Bloque 17: 1,000 productos revisados, 360 seleccionados
Bloque 18: 1,000 productos revisados, 341 seleccionados
B

In [8]:
print("Tipo de bloques_videojuegos:", type(bloques_videojuegos))
print("Número de bloques guardados:", len(bloques_videojuegos))

Tipo de bloques_videojuegos: <class 'list'>
Número de bloques guardados: 138


In [9]:
# Concatenar los bloques de videojuegos en un solo DataFrame
metadatos_videojuegos = pd.concat(
    bloques_videojuegos,
    ignore_index=True
)

In [10]:
print("Tipo del resultado:", type(metadatos_videojuegos))
print("Dimensiones:", metadatos_videojuegos.shape)

Tipo del resultado: <class 'pandas.DataFrame'>
Dimensiones: (46533, 7)


In [11]:
# Comprobación de duplicados en la columna "parent_asin"
numero_duplicados = (
    metadatos_videojuegos["parent_asin"]
    .duplicated()
    .sum()
)

print("Identificadores duplicados:", numero_duplicados)

Identificadores duplicados: 0


In [12]:
# Filtrado de duplicados en la columna "parent_asin" y reinicio del índice
metadatos_videojuegos = (
    metadatos_videojuegos
    .drop_duplicates(subset="parent_asin")
    .reset_index(drop=True)
)

In [13]:
# Cálculo del porcentaje de productos seleccionados
porcentaje_seleccionado = (
    len(metadatos_videojuegos)
    / total_productos_revisados
    * 100
)

print("RESULTADO DEL FILTRADO DE PRODUCTOS")
print("-----------------------------------")
print(f"Productos revisados: {total_productos_revisados:,}")
print(f"Registros seleccionados: {total_productos_seleccionados:,}")
print(f"Videojuegos únicos: {len(metadatos_videojuegos):,}")
print(f"Porcentaje seleccionado: {porcentaje_seleccionado:.2f} %")

RESULTADO DEL FILTRADO DE PRODUCTOS
-----------------------------------
Productos revisados: 137,269
Registros seleccionados: 46,533
Videojuegos únicos: 46,533
Porcentaje seleccionado: 33.90 %


In [14]:
muestra_videojuegos = (
    metadatos_videojuegos[
        ["title", "categories", "parent_asin"]
    ]
    .sample(
        n=min(30, len(metadatos_videojuegos)),
        random_state=42
    )
)

muestra_videojuegos

,title,categories,parent_asin
3288,Transformers: Dark of the Moon Decepticons - N...,"[Video Games, Legacy Systems, Nintendo Systems...",B004MXTZTS
24569,Nba 2K18 Legend Gold Edition - Xbox One,"[Video Games, Xbox One, Games]",B0741B5N1F
3227,Grand Theft Auto: Episodes from Liberty City [...,"[Video Games, PC, Games]",B004D4YDPC
37150,Fullmetal Alchemist Trading Card Game - Ninten...,"[Video Games, Legacy Systems, Nintendo Systems...",B000P97E8M
41612,Steep Gold Edition - PS4 [Digital Code],"[Video Games, PlayStation Digital Content, Games]",B01N1NBERC
37416,Return to Krondor,"[Video Games, PC, Games]",B000W2K876
26003,For Honor: Battle Pass - Year 4 Season 1 - PS4...,"[Video Games, PlayStation Digital Content, Games]",B08527ZXQ1
8913,Reel Deal Card Games 2011 [Download],"[Video Games, PC, Games]",B00571X5OU
2095,Cities Skylines (PS4),"[Video Games, PlayStation 4, Games]",B07D6X9QLW
17452,Call Of Duty: Ghosts /xbox One,"[Video Games, Xbox One, Games]",B00CM1KWAS


In [15]:
# Renombrar columnas para mayor claridad
metadatos_videojuegos = metadatos_videojuegos.rename(
    columns={
        "title": "product_title",
        "average_rating": "product_average_rating",
        "rating_number": "product_rating_number"
    }
)

In [ ]:
ruta_metadatos_filtrados = (
    RUTA_provisional / "metadatos_videojuegos.parquet"
)

metadatos_videojuegos.to_parquet(
    ruta_metadatos_filtrados,
    index=False
)

print("Archivo guardado en:")
print(ruta_metadatos_filtrados.resolve())

Archivo guardado en:
C:\Users\Paula María\Desktop\IS\5º CURSO\TFG\Trabajo-Fin-de-Grado\data\interim\metadatos_videojuegos.parquet


In [29]:
# Cálculo de identificadores únicos de videojuegos
ids_videojuegos = set(
    metadatos_videojuegos["parent_asin"]
    .dropna()
    .astype(str)
    .unique()
)

print(f"Identificadores de videojuegos: {len(ids_videojuegos):,}")

Identificadores de videojuegos: 46,533


In [ ]:
# Guardar los bloques de reseñas de videojuegos en archivos separados
RUTA_BLOQUES = (
    RUTA_provisional / "resenas_videojuegos_bloques"
)

RUTA_BLOQUES.mkdir(
    parents=True,
    exist_ok=True
)

total_resenas_revisadas = 0
total_resenas_seleccionadas = 0
numero_archivo = 0

In [31]:
# Bucle para leer el archivo de reseñas en bloques y filtrar las reseñas de videojuegos
# El algoritmo lee el archivo de reseñas en bloques de 100,000 registros, filtra las reseñas que corresponden a videojuegos y guarda cada bloque filtrado en un archivo Parquet separado.
for numero_bloque, bloque in enumerate(
    pd.read_json(
        ruta_resenas,
        lines=True,
        compression="gzip",
        chunksize=100000
    ),
    start=1
):
    total_resenas_revisadas += len(bloque)

    bloque["parent_asin"] = (
        bloque["parent_asin"]
        .astype(str)
    )

    seleccion = bloque[
        bloque["parent_asin"].isin(ids_videojuegos)
    ].copy()

    if not seleccion.empty:
        ruta_salida = RUTA_BLOQUES / (
            f"resenas_videojuegos_{numero_archivo:04d}.parquet"
        )

        seleccion.to_parquet(
            ruta_salida,
            index=False
        )

        total_resenas_seleccionadas += len(seleccion)
        numero_archivo += 1

    print(
        f"Bloque {numero_bloque}: "
        f"{len(bloque):,} revisadas, "
        f"{len(seleccion):,} seleccionadas"
    )

Bloque 1: 100,000 revisadas, 39,231 seleccionadas
Bloque 2: 100,000 revisadas, 32,353 seleccionadas
Bloque 3: 100,000 revisadas, 33,745 seleccionadas
Bloque 4: 100,000 revisadas, 35,227 seleccionadas
Bloque 5: 100,000 revisadas, 33,000 seleccionadas
Bloque 6: 100,000 revisadas, 34,064 seleccionadas
Bloque 7: 100,000 revisadas, 35,931 seleccionadas
Bloque 8: 100,000 revisadas, 41,028 seleccionadas
Bloque 9: 100,000 revisadas, 33,160 seleccionadas
Bloque 10: 100,000 revisadas, 34,812 seleccionadas
Bloque 11: 100,000 revisadas, 37,489 seleccionadas
Bloque 12: 100,000 revisadas, 33,658 seleccionadas
Bloque 13: 100,000 revisadas, 36,308 seleccionadas
Bloque 14: 100,000 revisadas, 33,566 seleccionadas
Bloque 15: 100,000 revisadas, 29,105 seleccionadas
Bloque 16: 100,000 revisadas, 31,627 seleccionadas
Bloque 17: 100,000 revisadas, 31,429 seleccionadas
Bloque 18: 100,000 revisadas, 32,649 seleccionadas
Bloque 19: 100,000 revisadas, 36,717 seleccionadas
Bloque 20: 100,000 revisadas, 26,618 sel

In [33]:
porcentaje_conservado = (
    total_resenas_seleccionadas
    / total_resenas_revisadas
    * 100
)

print("\nRESULTADO DEL FILTRADO DE RESEÑAS")
print("---------------------------------")
print(f"Reseñas revisadas: {total_resenas_revisadas:,}")
print(f"Reseñas seleccionadas: {total_resenas_seleccionadas:,}")
print(f"Porcentaje conservado: {porcentaje_conservado:.2f} %")
print(f"Archivos generados: {numero_archivo}")


RESULTADO DEL FILTRADO DE RESEÑAS
---------------------------------
Reseñas revisadas: 4,624,615
Reseñas seleccionadas: 1,502,718
Porcentaje conservado: 32.49 %
Archivos generados: 47
